## TODO
* Автоэнкодер
* Cross-domain
* Beam search with lp and cp
* SRU
* Визуализация Attn
* replace_unk по attention'у http://opennmt.net/OpenNMT/translation/unknowns/

## Tutorials
* http://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html
* https://github.com/spro/practical-pytorch/blob/master/seq2seq-translation/seq2seq-translation-batched.ipynb

## Articles
* Teaching neural networks to point to improve language modeling and translation: https://einstein.ai/research/teaching-neural-networks-to-point-to-improve-language-modeling-and-translation
* Training RNNs as Fast as CNNs : https://arxiv.org/abs/1709.02755
* Beam Search Strategies for Neural Machine Translation: https://arxiv.org/abs/1702.01806
* Unsupervised Machine Translation Using Monolingual Corpora Only: https://arxiv.org/abs/1711.00043
* Unsupervised Neural Machine Translation: https://arxiv.org/abs/1710.11041
* NIPS 2016 Tutorial: Generative Adversarial Networks: https://arxiv.org/pdf/1701.00160.pdf

## Repos
* https://github.com/facebookresearch/MUSE
* https://github.com/OpenNMT/OpenNMT-py

In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
from torch import optim
from collections import Counter, namedtuple
import pickle
import os
import re
import random
import time
import numpy as np
from typing import List, Tuple
from torch.nn.utils.rnn import pack_padded_sequence as pack
from torch.nn.utils.rnn import pad_packed_sequence as unpack
from collections import Counter

from utils.vocabulary import Vocabulary
from utils.tqdm import tqdm_open

use_cuda = torch.cuda.is_available()
print(use_cuda)

True


In [133]:
class Batch:
    def __init__(self, src_variable, tgt_variable, src_lengths, tgt_lengths):
        self.src_variable = src_variable
        self.tgt_variable = tgt_variable
        self.src_lengths = src_lengths
        self.tgt_lengths = tgt_lengths
        
    def cuda(self):
        return Batch(self.src_variable.cuda(), self.tgt_variable.cuda(), src_lengths, tgt_lengths)
    
    def __str__(self):
        return "Batch: " + str(self.src_variable) + ", " + str(self.tgt_variable) + ", " + str(self.src_lengths) + ", " + str(self.tgt_lengths)

In [3]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size, n_layers=3, dropout=0.1):
        super(EncoderRNN, self).__init__()
        
        num_directions = 2
        assert hidden_size % num_directions == 0
        hidden_size = hidden_size // num_directions
        
        self.embedding_dim = embedding_dim
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.dropout = dropout
       
        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_size, n_layers, dropout=dropout, bidirectional=True)
        
    def forward(self, input_seqs, input_lengths, hidden=None):
        embedded = self.embedding(input_seqs)
        packed = pack(embedded, input_lengths)
        outputs, hidden = self.rnn(packed, hidden)
        outputs, output_lengths = unpack(outputs)
        n = hidden[0].size(0)
        hidden = (torch.cat([hidden[0][0:n:2], hidden[0][1:n:2]], 2), torch.cat([hidden[1][0:n:2], hidden[1][1:n:2]], 2))
        return outputs, hidden

In [4]:
class Attn(nn.Module):
    def __init__(self, hidden_size):
        super(Attn, self).__init__()
        
        self.hidden_size = hidden_size
        self.attn = nn.Linear(hidden_size, hidden_size, bias=False)
        self.sm = nn.Softmax(dim=1)
        
        self.out = nn.Linear(hidden_size * 2, hidden_size, bias=False)
        self.tanh = nn.Tanh()

    def forward(self, decoder_rnn_output, encoder_outputs):
        max_len = encoder_outputs.size(0)
        batch_size = encoder_outputs.size(1)
        
        decoder_rnn_output = decoder_rnn_output.transpose(0, 1)
        energy = self.attn(encoder_outputs).view(batch_size, self.hidden_size, max_len)
        attn_energies = decoder_rnn_output.bmm(energy).transpose(0, 1).squeeze(0)  # S = B x L
        
        attn_weights = self.sm(attn_energies).unsqueeze(1) # S = B x 1 x L
        encoder_context = attn_weights.bmm(encoder_outputs.transpose(0, 1)).transpose(0, 1) # S = 1 x B x N
        
        concat_context = torch.cat([encoder_context, decoder_rnn_output.transpose(0, 1)], 2)
        context = self.tanh(self.out(concat_context))

        return context, attn_weights.squeeze(1)

In [5]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_size, output_size, n_layers=3, dropout=0.1, max_length=50):
        super(AttnDecoderRNN, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        self.max_length = max_length
        
        self.embedding = nn.Embedding(output_size, embedding_dim)
        self.attn = Attn(hidden_size)
        self.rnn = nn.LSTM(hidden_size + embedding_dim, hidden_size, n_layers, dropout=dropout)

    def step(self, input_seq, hidden, encoder_outputs, context):
        # hidden: S = n_layers x B x N
        # encoder_outputs: S = L x B x N 
        embedded = self.embedding(input_seq).unsqueeze(0) # S = 1 x B x E
    
        # Combine embedded input word and attended context, run through RNN (input feeding)
        rnn_input = torch.cat((embedded, context), 2)
        output, hidden = self.rnn(rnn_input, hidden)
        
         # Calculate attention weights and apply to encoder outputs
        output, attn_weights = self.attn(output, encoder_outputs) 
        # output: # S = 1 x B x N
        
        # Return final output, hidden state, and attention weights (for visualization)
        return output, hidden, attn_weights
    
    def init_state(self, batch_size):
        initial_input = Variable(torch.LongTensor([1 for _ in range(batch_size)]), requires_grad=False)
        initial_input = initial_input.cuda() if use_cuda else initial_input
        
        initial_context = Variable(torch.zeros(batch_size, self.hidden_size), requires_grad=False).unsqueeze(0)
        initial_context = initial_context.cuda() if use_cuda else initial_context
        
        return initial_input, initial_context
    
    def forward(self, inputs, input_lengths, hidden, encoder_outputs, initial_input, initial_context):
        batch_size = encoder_outputs.size(1)
        max_input_length = max(input_lengths)
        max_encoder_length = encoder_outputs.size(0)
        
        outputs = Variable(torch.zeros(max_input_length + 1, batch_size, self.hidden_size), requires_grad=False)
        outputs = outputs.cuda() if use_cuda else outputs
        
        attn_weights = Variable(torch.zeros(max_input_length + 1, batch_size, max_encoder_length), requires_grad=False)
        attn_weights = attn_weights.cuda() if use_cuda else attn_weights
        
        context = initial_context
        for t in range(max_input_length + 1):
            if t != 0:
                current_input = inputs[t-1]
            else:
                current_input = initial_input
            context, hidden, attn = self.step(current_input, hidden, encoder_outputs, context)
            outputs[t] = context
            attn_weights[t] = attn
        
        return outputs, hidden, attn_weights

In [6]:
class Generator(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Generator, self).__init__()
        
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        self.out = nn.Linear(hidden_size, output_size)
        self.sm = nn.LogSoftmax(dim=1)
    
    def forward(self, inputs):
        return self.sm(self.out(inputs))

In [7]:
class Seq2SeqAttn(nn.Module):
    def __init__(self, input_embedding_dim, output_embedding_dim, input_size, output_size, hidden_size, 
                 encoder_n_layers=3, decoder_n_layers=3, dropout=0.1, max_length=50):
        super(Seq2SeqAttn, self).__init__()
        
        self.input_embedding_dim = input_embedding_dim
        self.output_embedding_dim = output_embedding_dim
        self.input_size = input_size
        self.output_size = output_size
        self.hidden_size = hidden_size
        self.encoder_n_layers = encoder_n_layers
        self.decoder_n_layers = decoder_n_layers
        self.dropout = dropout
        self.max_length = max_length
        
        self.encoder = EncoderRNN(input_size, input_embedding_dim, hidden_size, dropout=dropout, 
                                  n_layers=encoder_n_layers)
        self.decoder = AttnDecoderRNN(output_embedding_dim, hidden_size, output_size, dropout=dropout, 
                                      max_length=max_length, n_layers=decoder_n_layers)
        self.generator = Generator(hidden_size, output_size)
    
    def forward(self, batch: Batch, batch_size, criterion):
        input_variable = batch.src_variable
        target_variable = batch.tgt_variable
        if use_cuda:
            input_variable = input_variable.cuda()
            target_variable = target_variable.cuda()
        input_lengths = batch.src_lengths
        target_lengths = batch.tgt_lengths

        encoder_output, encoder_hidden = self.encoder(input_variable, input_lengths, None)
        initial_input, initial_context = self.decoder.init_state(batch_size)
        decoder_output, _, attn_weights = self.decoder(target_variable, target_lengths, encoder_hidden, encoder_output, initial_input, initial_context)
        
        loss = 0
        max_target_length = max(target_lengths)
        for t in range(max_target_length):
            scores = self.generator(decoder_output[t])
            loss += criterion(scores, target_variable[t])
        return loss, attn_weights

In [12]:
# lang1 = 'en'
# lang2 = 'de'
# input_embedding_dim = 300
# output_embedding_dim = 300
# hidden_size = 500
# encoder_n_layers = 2
# decoder_n_layers = 2
# encoder_dropout = 0.2
# decoder_dropout = 0.2
# max_length = 50
# input_vocabulary, output_vocabulary = Vocabulary(lang1), Vocabulary(lang2)
# DATASET = "/media/yallen/My Passport/Projects/UNMT/train"
# INPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/train.en"
# OUTPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/train.de"
# INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/train-sorted.en"
# OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/train-sorted.de"
# VAL_INPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/val.en"
# VAL_OUTPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/val.de"
# VAL_INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/val-sorted.en"
# VAL_OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/val-sorted.de"
# pairs = [(INPUT_FILENAME, OUTPUT_FILENAME)]

In [13]:
# def sort(src_filename, src_filename_sorted, tgt_filename, tgt_filename_sorted):
#     with open(src_filename, "r", encoding="utf-8") as r1, open(tgt_filename, "r", encoding="utf-8") as r2:
#         lines = list(zip(r1.readlines(), r2.readlines()))
#         lines = sorted(lines, key=lambda x: len(x[0]))
#         print(lines[10], lines[-10])
#     with open(src_filename_sorted, "w", encoding="utf-8") as w1, open(tgt_filename_sorted, "w", encoding="utf-8") as w2:
#         for line in lines:
#             w1.write(line[0])
#             w2.write(line[1])
# sort(INPUT_FILENAME_RAW, INPUT_FILENAME, OUTPUT_FILENAME_RAW, OUTPUT_FILENAME)
# sort(VAL_INPUT_FILENAME_RAW, VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME_RAW, VAL_OUTPUT_FILENAME)

In [14]:
# def prepare(pair_filenames: List[Tuple[str, str]], lang1: str, lang2: str):
#     if input_vocabulary.is_empty() or output_vocabulary.is_empty():
#         for lang1_filename, lang2_filename in pair_filenames:
#             with tqdm_open(lang1_filename, encoding="utf-8") as r1, open(lang2_filename, "r", encoding="utf-8") as r2:
#                 for input_sentence, output_sentence in zip(r1, r2):
#                     input_vocabulary.add_sentence(input_sentence.strip())
#                     output_vocabulary.add_sentence(output_sentence.strip())
#         input_vocabulary.shrink(100000)
#         output_vocabulary.shrink(100000)
#         print(input_vocabulary.word2count.most_common(50))
#         print(output_vocabulary.word2count.most_common(50))
#         input_vocabulary.save()
#         output_vocabulary.save()
# prepare(pairs, "en", "de")

In [134]:
def indices_from_sentence(sentence: str, vocabulary: Vocabulary):
    return [vocabulary.get_index(word) for word in sentence.split(' ')] + [vocabulary.get_eos()]


def pad_seq(seq: List[int], vocabulary: Vocabulary, max_length: int):
    seq += [vocabulary.get_pad() for _ in range(max_length - len(seq))]
    return seq

class BatchGenerator:
    def __init__(self, pair_filenames: List[Tuple[str, str]], batch_size: int, max_sentence_len: int,
                 input_vocabulary: Vocabulary, output_vocabulary: Vocabulary, use_cuda: bool=True):
        self.pair_filenames = pair_filenames  # type: List[Tuple[str, str]]
        self.batch_size = batch_size  # type: int
        self.max_sentence_len = max_sentence_len  # type: int
        self.input_vocabulary = input_vocabulary
        self.output_vocabulary = output_vocabulary
        self.use_cuda = use_cuda

    def __iter__(self):
        for lang1_filename, lang2_filename in self.pair_filenames:
            input_seqs = []
            output_seqs = []
            with tqdm_open(lang1_filename, encoding='utf-8') as r1, open(lang2_filename, "r", encoding="utf-8") as r2:
                for input_sentence, output_sentence in zip(r1, r2):
                    input_sentence = input_sentence.strip()
                    output_sentence = output_sentence.strip()

                    input_sentence = indices_from_sentence(input_sentence, self.input_vocabulary)
                    output_sentence = indices_from_sentence(output_sentence, self.output_vocabulary)
                    
                    if len(input_sentence) >= self.max_sentence_len - 1 or len(output_sentence) >= self.max_sentence_len - 1:
                        continue

                    input_seqs.append(input_sentence)
                    output_seqs.append(output_sentence)
                    if len(input_seqs) == self.batch_size:
                        yield self.__process(input_seqs, output_seqs)
                        input_seqs = []
                        output_seqs = []
            if len(input_seqs) == self.batch_size:
                yield self.__process(input_seqs, output_seqs)

    def __process(self, input_seqs, output_seqs):
        input_padded, output_padded, input_lengths, output_lengths = self.__pad(input_seqs, output_seqs)
        input_variable, output_variable = self.__to_tensor(input_padded, output_padded)
        return Batch(input_variable, output_variable, input_lengths, output_lengths)

    def __pad(self, input_seqs, output_seqs):
        seq_pairs = sorted(zip(input_seqs, output_seqs), key=lambda p: len(p[0]), reverse=True)
        input_seqs, target_seqs = zip(*seq_pairs)
        input_lengths = [len(s) for s in input_seqs]
        input_padded = [pad_seq(s, self.input_vocabulary, max(input_lengths)) for s in input_seqs]
        output_lengths = [len(s) for s in target_seqs]
        output_padded = [pad_seq(s, self.output_vocabulary, max(output_lengths)) for s in target_seqs]
        return input_padded, output_padded, input_lengths, output_lengths

    def __to_tensor(self, input_padded, output_padded):
        # Turn padded arrays into (batch_size x max_len) tensors, transpose into (max_len x batch_size)
        input_variable = Variable(torch.LongTensor(input_padded), requires_grad=False).transpose(0, 1)
        output_variable = Variable(torch.LongTensor(output_padded), requires_grad=False).transpose(0, 1)
        return input_variable, output_variable

In [16]:
def train_batch(model, optimizer, criterion, batch: Batch, batch_size):
    target_lengths = batch.output_lengths
    target_tokens_count = sum(target_lengths)
    optimizer.zero_grad()
    loss, _ = model(batch, batch_size, criterion)
    loss.backward()
    nn.utils.clip_grad_norm(model.parameters(), 5)
    optimizer.step()
    return loss.data[0] / target_tokens_count

def validate_batch(model, criterion, batch: Batch, batch_size):
    target_lengths = batch.output_lengths
    target_tokens_count = sum(target_lengths)
    loss, _ = model(batch, batch_size, criterion)
    return loss.data[0] / target_tokens_count

def train(model, optimizer, train_pair_filenames: List[Tuple[str, str]], val_pair_filenames: List[Tuple[str, str]],
          big_epochs: int, print_every=3000, save_every=3000, batch_size=32):
    weight = torch.ones(output_vocabulary.size())
    weight[output_vocabulary.get_pad()] = 0
    weight = weight.cuda() if use_cuda else weight
    criterion = nn.NLLLoss(weight, size_average=False)
    train_batch_generator = BatchGenerator(train_pair_filenames, batch_size, max_length,
                                           input_vocabulary, output_vocabulary, use_cuda)
    train_batches = []
    i = 0
    for batch in train_batch_generator:
#         if i == 10000:
#             break
        train_batches.append(batch)
        i += 1
    count_batches = len(train_batches)
    
    val_batch_generator = BatchGenerator(val_pair_filenames, batch_size, max_length,
                                         input_vocabulary, output_vocabulary, use_cuda)
    val_batches = []
    for batch in val_batch_generator:
        val_batches.append(batch)
    val_perm = np.random.permutation(len(val_batches))[:1000]
    val_batches = [batch for i, batch in enumerate(val_batches) if i in val_perm]
    
    print(model)
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print("Params: ", params)
    print("Input:", train_batches[0].src_variable)
    print("Output:", train_batches[0].tgt_variable)
    
    for big_epoch in range(big_epochs):
        timer = time.time()
        print_loss_total = 0
        count_tokens = 0
        perm = np.random.permutation(count_batches)
        for epoch, batch_index in enumerate(perm):
            batch = train_batches[batch_index]
            loss = train_batch(model, optimizer, criterion, batch, batch_size)
            
            print_loss_total += loss
            count_tokens += sum(batch.input_lengths)
            if epoch % save_every == 0 and epoch != 0:
                save(model, optimizer, "model.pt")
            if epoch % print_every == 0 and epoch != 0:
                val_loss = 0
                for val_batch in val_batches:
                    val_loss += validate_batch(model, criterion, val_batch, batch_size)
                val_loss /= len(val_batches)
                print_loss_avg = print_loss_total / print_every
                print_loss_total = 0
                diff = time.time() - timer
                timer = time.time()
                src_speed = count_tokens / diff
                print('%s big epoch, %s/%s, %s src tok/s, %s sec, %.4f loss, %.4f val loss' % 
                      (big_epoch, epoch, count_batches, src_speed, diff, print_loss_avg, val_loss))
                count_tokens = 0

In [17]:
def save_model(module, optimizer, filename):
    state_dict = module.state_dict()
    for key in state_dict.keys():
        state_dict[key] = state_dict[key].cpu()
    torch.save({
        'state_dict': state_dict,
        'optimizer' : optimizer.state_dict(),
    },filename)

def save(model, optimizer, model_filename):
    save_model(model, optimizer, model_filename)
    
def load(model, optimizer, model_filename):
    checkpoint = torch.load(model_filename)
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    return model, optimizer

In [18]:
# print("Building...")
# model = Seq2SeqAttn(input_embedding_dim, output_embedding_dim, input_vocabulary.size(), output_vocabulary.size(), hidden_size)
# model = model.cuda() if use_cuda else model
# optimizer = optim.SGD([p for p in model.parameters() if p.requires_grad], lr=1)

In [19]:
# print("Loading...")
# model, optimizer = load(model, optimizer, "model.pt")

In [20]:
# print("Training...")
# train(model, optimizer, pairs, [(VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME), ], 
#       big_epochs=5, print_every=1000, save_every=1000, batch_size=64)

In [21]:
def translate(model, sentence):
    batch_size = 64
    result = [[] for i in range(batch_size)]
    
    output_variable = Variable(torch.zeros(max_length, batch_size)).type(torch.LongTensor)
    output_variable = output_variable.cuda() if use_cuda else output_variable
    
    indices = indices_from_sentence(sentence, input_vocabulary)
    input_variable = Variable(torch.zeros(batch_size, len(indices))).type(torch.LongTensor)
    indices = Variable(torch.LongTensor(indices))
    input_variable[0] = indices
    for i in range(1, batch_size):
        input_variable[i, 0] = input_vocabulary.get_eos()
    input_variable = input_variable.transpose(0, 1)
    input_variable = input_variable.cuda() if use_cuda else input_variable
    
    input_lengths = [len(indices)]
    input_lengths += [1 for _ in range(batch_size-1)]
    
    encoder_output, encoder_hidden = model.encoder(input_variable, input_lengths, None)
    initial_input, initial_context = model.decoder.init_state(batch_size)
    
    for t in range(max_length):
        decoder_output, decoder_hidden, attn_weights = model.decoder(output_variable, [t+1 for i in range(batch_size)],
                                                                     encoder_hidden, encoder_output, initial_input, initial_context)
        scores = model.generator(decoder_output[t])
        for i in range(1):
            topv, topi = scores.data[i].topk(1)
            ni = topi[0]
            output_variable[t, i] = topi
            if ni == output_vocabulary.get_eos() or ni == output_vocabulary.get_pad():
                result[i].append('<EOS>')
            else:
                word = output_vocabulary.index2word[ni]
                result[i].append(word)
        for i, sentence in enumerate(result):
            end = sentence.index('<EOS>') if '<EOS>' in sentence else len(sentence)
            result[i] = sentence[:end]
    return result[0]

In [66]:
translate(model, "I object to yet another increase in Parliament &apos;s budget .")

['Ich',
 'lehne',
 'noch',
 'ein',
 'weiteres',
 'Parlament',
 'im',
 'Haushalt',
 'des',
 'Parlaments',
 'ein',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 '.',
 'und',
 'noch',
 'neu',
 'zu',
 'erreichen',
 '.',
 '.',
 '.',
 '.',
 'und',
 'neu',
 'zu',
 'erreichen']

In [35]:
batch_size = 64
result = [[] for i in range(batch_size)]

val_batch_generator = BatchGenerator([(VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME), ], batch_size, max_length,
                                         input_vocabulary, output_vocabulary, use_cuda)
val_batches = []
for batch in val_batch_generator:
    val_batches.append(batch)
val_perm = np.random.permutation(len(val_batches))[:1000]
val_batches = [batch for i, batch in enumerate(val_batches) if i in val_perm]   
batch = val_batches[900]

input_variable = batch.input_variable
if use_cuda:
    input_variable = input_variable.cuda()
print(input_variable)

output_variable = Variable(torch.zeros(max_length, batch_size)).type(torch.LongTensor)
output_variable = output_variable.cuda() if use_cuda else output_variable
print(input_variable.size())

encoder_output, encoder_hidden = model.encoder(input_variable, batch.input_lengths, None)
initial_input, initial_context = model.decoder.init_state(batch_size)
max_target_length = max(batch.output_lengths)
for t in range(max_target_length):
    decoder_output, decoder_hidden, attn_weights = model.decoder(output_variable, [t+1 for i in range(batch_size)],
                                                                 encoder_hidden, encoder_output, initial_input, initial_context)
    scores = model.generator(decoder_output[t])
    for i in range(batch_size):
        topv, topi = scores.data[i].topk(1)
        ni = topi[0]
        output_variable[t, i] = topi
        if ni == output_vocabulary.get_eos() or ni == output_vocabulary.get_pad():
            result[i].append('<EOS>')
        else:
            word = output_vocabulary.index2word[ni]
            result[i].append(word)
for sentence in result:
    end = sentence.index('<EOS>') if '<EOS>' in sentence else len(sentence)
    sentence = sentence[:end]
    print(" ".join(sentence))

val-sorted.en: 100%|█████████▉| 23.9M/23.9M [00:07<00:00, 3.36MB/s]


Variable containing:
   540     10     10  ...     155      6  10582
   223   2889      7  ...     171    664   3521
   108      5     46  ...      29    210   2677
        ...            ⋱           ...         
    60      2      0  ...       0      0      0
   540      0      0  ...       0      0      0
     2      0      0  ...       0      0      0
[torch.cuda.LongTensor of size 33x64 (GPU 0)]

torch.Size([33, 64])
Variable containing:
   540     10     10  ...     155      6  10582
   223   2889      7  ...     171    664   3521
   108      5     46  ...      29    210   2677
        ...            ⋱           ...         
    60      2      0  ...       0      0      0
   540      0      0  ...       0      0      0
     2      0      0  ...       0      0      0
[torch.LongTensor of size 33x64]

# Jetzt die kleinen Pillen , die Farbe und du # # Wenn du ihn fragst , wenn sie es tun , wenn es dir was passiert , sag ich dir ,
Ich dachte , das wissen wir , wenn die Kinder es liebe

In [67]:
VAL_FILENAME = "src.txt"
OUTPUT_FILENAME = "pred.txt"
with open(VAL_FILENAME, "r", encoding='utf-8') as r:
    with open(OUTPUT_FILENAME, "w", encoding='utf-8') as w:
        for line in r:
            line = line.strip()
            translation = translate(model, line)
            w.write(" ".join(translation) + "\n")
!perl multi-bleu.perl ref.txt < pred.txt

BLEU = 3.80, 13.9/4.7/2.3/1.4 (BP=1.000, ratio=2.919, hyp_len=5330, ref_len=1826)
It is in-advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


In [121]:
# M1
class NNModel:
    def __init__(self, model):
        self.model = model
    
    def translate(self, sentence):
        return translate(self.model, sentence)

class WordByWordModel:
    def __init__(self, bilingual_dict_filename):
        self.bilingual_dict_filename = bilingual_dict_filename
        self.src_vocabulary = Vocabulary(language="en")
        self.tgt_vocabulary = Vocabulary(language="de")
        self.src2tgt = {0:0, 1:1, 2:2, 3:3}
        self.tgt2src = {0:0, 1:1, 2:2, 3:3}
        with open(self.bilingual_dict_filename, "r", encoding='utf-8') as r:
            for line in r:
                src_word, tgt_word = line.strip().split()
                self.src_vocabulary.add_word(src_word)
                self.tgt_vocabulary.add_word(tgt_word)
                src_index = self.src_vocabulary.get_index(src_word)
                tgt_index = self.tgt_vocabulary.get_index(tgt_word)
                self.src2tgt[src_index] = tgt_index
                self.tgt2src[tgt_index] = src_index
                
    def translate_src2tgt(self, indices):
        result = []
        for src_index in indices:
            if src_index in self.src2tgt:
                result.append(self.src2tgt[src_index])
            else:
                result.append(self.tgt_vocabulary.get_ukn())
        return result
    
    def translate_tgt2src(self, indices):
        result = []
        for tgt_index in indices:
            if tgt_index in self.tgt2src:
                result.append(self.tgt2src[tgt_index])
            else:
                result.append(self.src_vocabulary.get_ukn())
        return result
    
    def translate_src2tgt_sentence(self, sentence):
        indices = []
        for word in sentence.split():
            word = word.lower()
            indices.append(self.src_vocabulary.get_index(word))
        result = self.translate_src2tgt(indices)
        result = [self.tgt_vocabulary.get_word(i) for i in result]
        result.append("<EOS>")
        return result
    
    def translate_tgt2src_sentence(self, sentence):
        indices = []
        for word in sentence.split():
            word = word.lower()
            indices.append(self.tgt_vocabulary.get_index(word))
        result = self.translate_tgt2src(indices)
        result = [self.src_vocabulary.get_word(i) for i in result]
        result.append("<EOS>")
        return result

BILINGUAL_DICT = "models/en-de.txt"
M0 = WordByWordModel(BILINGUAL_DICT)
current_model = M0

In [30]:
# M1 evaluate
VAL_FILENAME = "src.txt"
OUTPUT_FILENAME = "pred.txt"
with open(VAL_FILENAME, "r", encoding='utf-8') as r:
    with open(OUTPUT_FILENAME, "w", encoding='utf-8') as w:
        for line in r:
            line = line.strip()
            translation = current_model.translate_src2tgt_sentence(line)
            w.write(" ".join(translation[:-1]) + "\n")
!perl multi-bleu.perl -lc ref.txt < pred.txt

Use of uninitialized value in division (/) at multi-bleu.perl line 139, <STDIN> line 162.
BLEU = 0.00, 11.9/1.2/0.3/0.0 (BP=1.000, ratio=1.101, hyp_len=2010, ref_len=1826)
It is in-advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


In [26]:
class Discriminator(nn.Module):
    def __init__(self, max_length, encoder_hidden_size, hidden_size=1024, n_layers=3, activation=F.leaky_relu):
        super(Attn, self).__init__()
        
        self.encoder_hidden_size = encoder_hidden_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.activation = activation
        
        self.layers = []
        self.layers.append(nn.Linear(encoder_hidden_size*max_length, hidden_size))
        for i in range(n_layers-1):
            self.layers.append = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, 2)

    def forward(self, encoder_output):
        output = encoder_output.view(-1) # S = max_length * encoder_hidden_size
        for i in range(n_layers):
            output = self.layers[i](output)
            output = self.activation(output)
        return F.log_softmax(self.out(output), dim=1)

In [136]:
class UNMT(nn.Module):
    def __init__(self, embedding_dim, src_vocabulary, tgt_vocabulary, hidden_size,
                 encoder_n_layers=3, decoder_n_layers=3, dropout=0.1, max_length=50):
        super(Autoencoder, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.src_size = src_vocabulary.size()
        self.tgt_size = tgt_vocabulary.size()
        self.hidden_size = hidden_size
        self.encoder_n_layers = encoder_n_layers
        self.decoder_n_layers = decoder_n_layers
        self.dropout = dropout
        self.max_length = max_length
        self.src_vocabulary = src_vocabulary
        self.tgt_vocabulary = tgt_vocabulary
        
        self.src_encoder = EncoderRNN(src_size, embedding_dim, hidden_size, dropout=dropout, 
                                      n_layers=encoder_n_layers)
        self.tgt_encoder = EncoderRNN(tgt_size, embedding_dim, hidden_size, dropout=dropout, 
                                      n_layers=encoder_n_layers)
        self.src_decoder = AttnDecoderRNN(embedding_dim, hidden_size, src_size, dropout=dropout, 
                                          max_length=max_length, n_layers=decoder_n_layers)
        self.tgt_decoder = AttnDecoderRNN(embedding_dim, hidden_size, tgt_size, dropout=dropout, 
                                          max_length=max_length, n_layers=decoder_n_layers)
        self.src_generator = Generator(hidden_size, src_size)
        self.tgt_generator = Generator(hidden_size, tgt_size)
        self.discriminator = Discriminator(self.max_length, self.hidden_size)
    
    def forward(self, batch: Batch, noisy_batch: Batch, translated_noisy_batch: Batch, 
                batch_size, criterion, src_vocabulary, tgt_vocabulary):
        if use_cuda:
            batch = batch.cuda()
            noisy_batch = noisy_batch.cuda()
            translated_noisy_batch = translated_noisy_batch.cuda()
        
        src_adv_loss, src_auto_loss = \
            self.auto_encoder_decoder_run(self.src_encoder, self.src_decoder, self.src_generator, criterion, 
                                          noisy_batch.src_variable, noisy_batch.src_lengths, batch_size, lang="src")
            
        tgt_adv_loss, tgt_auto_loss = \
            self.auto_encoder_decoder_run(self.tgt_encoder, self.tgt_decoder, self.tgt_generator, criterion, 
                                          noisy_batch.tgt_variable, noisy_batch.tgt_lengths, batch_size, lang="tgt")
            
        cd_tgt_adv_loss, cd_tgt_cd_loss = \
            self.cd_encoder_decoder_run(self.src_encoder, self.tgt_decoder, self.tgt_generator, criterion, 
                                        translated_noisy_batch.tgt_variable, translated_noisy_batch.tgt_lengths, 
                                        batch.tgt_variable, batch_size, lang="tgt")
        
        cd_src_adv_loss, cd_src_cd_loss = \
            self.cd_encoder_decoder_run(self.tgt_encoder, self.src_decoder, self.src_generator, criterion, 
                                        translated_noisy_batch.src_variable, translated_noisy_batch.src_lengths, 
                                        batch.src_variable, batch_size, lang="src")
        
        return sum([src_adv_loss, src_auto_loss, tgt_adv_loss, tgt_auto_loss, 
                    cd_tgt_adv_loss, cd_tgt_cd_loss, cd_src_adv_loss, cd_src_cd_loss])
    
    def translate_src2tgt(self, indices):
        
    
    def translate_tgt2src(self, indices):
        result = [[] for i in range(batch_size)]
        
        output_variable = Variable(torch.zeros(self.max_length, self.batch_size)).type(torch.LongTensor)
        output_variable = output_variable.cuda() if use_cuda else output_variable
        
    def translate(self, variable, lengths, encoder, decoder, generator):
        encoder_output, encoder_hidden = model.encoder(variable, lengths, None)
        initial_input, initial_context = model.decoder.init_state(batch_size)

        for t in range(max_length):
            decoder_output, decoder_hidden, attn_weights = model.decoder(output_variable, [t+1 for i in range(batch_size)],
                                                                         encoder_hidden, encoder_output, initial_input, initial_context)
            scores = model.generator(decoder_output[t])
            for i in range(1):
                topv, topi = scores.data[i].topk(1)
                ni = topi[0]
                output_variable[t, i] = topi
                if ni == output_vocabulary.get_eos() or ni == output_vocabulary.get_pad():
                    result[i].append('<EOS>')
                else:
                    word = output_vocabulary.index2word[ni]
                    result[i].append(word)
            for i, sentence in enumerate(result):
                end = sentence.index('<EOS>') if '<EOS>' in sentence else len(sentence)
                result[i] = sentence[:end]
        return result[0]
        
        
    def auto_encoder_decoder_run(self, encoder, decoder, generator, criterion, variable, 
                                 lengths, batch_size, lang="src"):
        
        encoder_output, encoder_hidden = encoder(variable, lengths, None)
        
        # Adversarial part
        adv_criterion = nn.NLLLoss()
        log_proba = self.discriminator(encoder_output)
        if lang == "src":
            target_variable = Variable(torch.LongTensor([0, 1]))
        else:
            target_variable = Variable(torch.LongTensor([1, 0]))
        adv_loss = adv_criterion(log_proba, target_variable)
        
        # Auto part
        initial_input, initial_context = decoder.init_state(batch_size)
        decoder_output, _, _ = decoder(variable, lengths, encoder_hidden, encoder_output, 
                                       initial_input, initial_context)
        auto_loss = 0
        max_length = max(lengths)
        for t in range(max_length):
            scores = generator(decoder_output[t])
            auto_loss += criterion(scores, variable[t])
            
        return adv_loss, auto_loss

    def cd_encoder_decoder_run(self, encoder, decoder, generator, criterion, variable, lengths, 
                               gt_variable, batch_size, lang="src"):
        encoder_output, encoder_hidden = encoder(variable, lengths, None)
        
        # Adversarial part
        adv_criterion = nn.NLLLoss()
        log_proba = self.discriminator(encoder_output)
        if lang == "src":
            target_variable = Variable(torch.LongTensor([1, 0]))
        else:
            target_variable = Variable(torch.LongTensor([0, 1]))
        adv_loss = adv_criterion(log_proba, target_variable)
        
        # Cross-domain part
        initial_input, initial_context = decoder.init_state(batch_size)
        decoder_output, _, _ = decoder(variable, lengths, encoder_hidden, encoder_output, 
                                       initial_input, initial_context)
        
        cd_loss = 0
        max_length = max(lengths)
        for t in range(max_length):
            scores = generator(decoder_output[t])
            cd_loss += criterion(decoder_output, gt_variable[t])
        
        return adv_loss, cd_loss

In [122]:
class GlobalState:
    def __init__(self, bilingual_dict="models/en-de.txt", src_embeddings="models/wiki.multi.en.vec", 
                 tgt_wmbeddings="models/wiki.multi.de.vec", batch_size=64):
        self.current_model = WordByWordModel(bilingual_dict)
        self.src_vocabulary = self.current_model.src_vocabulary
        self.tgt_vocabualry = self.current_model.tgt_vocabulary
        self.batch_size = batch_size
        self.max_length = 50
        self.discriminator_optimizer = None
        self.main_optimizer = None
        
        weight = torch.ones(self.tgt_vocabulary.size())
        weight[self.tgt_vocabulary.get_pad()] = 0
        weight = weight.cuda() if use_cuda else weight
        self.tgt_criterion = nn.NLLLoss(weight, size_average=False)
        
        weight = torch.ones(self.src_vocabulary.size())
        weight[self.src_vocabulary.get_pad()] = 0
        weight = weight.cuda() if use_cuda else weight
        self.src_criterion = nn.NLLLoss(weight, size_average=False)

        
    def train(self, train_pair_filenames: List[Tuple[str, str]], val_pair_filenames: List[Tuple[str, str]],
              big_epochs: int, print_every=3000, save_every=3000, batch_size=32):
        model = UNMT(300, self.src_vocabulary, self.tgt_vocabulary, 500)
        self.discriminator_optimizer = optim.SGD(model.discriminator.parameters(), lr=1)
        self.main_optimizer = optim.SGD(model.parameters(), lr=1)
        
        train_batch_generator = BatchGenerator(train_pair_filenames, batch_size, max_length, 
                                               self.src_vocabulary, self.tgt_vocabulary, use_cuda)
        train_batches = []
        for batch in train_batch_generator:
            train_batches.append(batch)
        count_batches = len(train_batches)

        val_batch_generator = BatchGenerator(val_pair_filenames, batch_size, max_length, 
                                             self.src_vocabulary, self.tgt_vocabulary, use_cuda)
        val_batches = []
        for batch in val_batch_generator:
            val_batches.append(batch)
        val_perm = np.random.permutation(len(val_batches))[:1000]
        val_batches = [val_batches[index] for index in val_perm]

        print(model)
        
        model_parameters = filter(lambda p: p.requires_grad, model.parameters())
        params = sum([np.prod(p.size()) for p in model_parameters])
        print("Params: ", params)
        
        print("Input:", train_batches[0].src_variable)
        print("Output:", train_batches[0].tgt_variable)

        for big_epoch in range(big_epochs):
            timer = time.time()
            print_loss_total = 0
            count_tokens = 0
            perm = np.random.permutation(count_batches)
            for epoch, batch_index in enumerate(perm):
                batch = train_batches[batch_index]
                discrimintor_loss, main_loss = self.train_batch(model, batch, batch_size)
                print(discrimintor_loss, main_loss)

                print_loss_total += main_loss
                count_tokens += sum(batch.input_lengths)
                if epoch % save_every == 0 and epoch != 0:
                    save(model, optimizer, "model.pt")
                if epoch % print_every == 0 and epoch != 0:
                    val_loss = 0
                    for val_batch in val_batches:
                        val_loss += validate_batch(model, criterion, val_batch, batch_size)
                    val_loss /= len(val_batches)
                    print_loss_avg = print_loss_total / print_every
                    print_loss_total = 0
                    diff = time.time() - timer
                    timer = time.time()
                    src_speed = count_tokens / diff
                    print('%s big epoch, %s/%s, %s src tok/s, %s sec, %.4f loss, %.4f val loss' % 
                          (big_epoch, epoch, count_batches, src_speed, diff, print_loss_avg, val_loss))
                    count_tokens = 0
        
        
    def train_batch(self, model, batch: Batch, batch_size):
        noisy_batch = prepare_noisy_input(batch)
        translated_noisy_batch = prepare_translated_noisy_input(batch)
        
        # Disciminator step
        batch = batch.cuda()
        self.discriminator_optimizer.zero_grad()
        adv_criterion = nn.NLLLoss()
        
        src_encoder_output, _ = model.src_encoder(batch.src_variable, batch.src_lengths, None)
        log_proba = self.discriminator(src_encoder_output)
        src_variable = Variable(torch.LongTensor([1, 0]))
        src_adv_loss = adv_criterion(log_proba, src_variable)
        
        tgt_encoder_output, _ = model.tgt_encoder(batch.tgt_variable, batch.tgt_lengths, None)
        log_proba = self.discriminator(tgt_encoder_output)
        tgt_variable = Variable(torch.LongTensor([0, 1]))
        tgt_adv_loss = adv_criterion(log_proba,  tgt_variable)
        
        discriminator_loss = src_adv_loss + tgt_adv_loss
        discriminator_loss.backward()
        nn.utils.clip_grad_norm(model.discriminator.parameters(), 5)
        self.discriminator_optimizer.step()
        
        # Main step
        noisy_batch = noisy_batch.cuda()
        translated_noisy_batch = translated_noisy_batch.cuda()
        self.main_optimizer.zero_grad()
        loss = model(batch, noisy_batch, translated_noisy_batch, self.batch_size, 
                     criterion, self.src_vocabulary, self.tgt_vocabulary)
        loss.backward()
        nn.utils.clip_grad_norm(model.parameters(), 5)
        self.main_optimizer.step()
        
    def get_variable(self, sentence, vocabulary):
        indices = indices_from_sentence(sentence, self.src_vocabulary)
        variable = Variable(torch.zeros(self.batch_size, len(indices))).type(torch.LongTensor)
        indices = Variable(torch.LongTensor(indices))
        variable[0] = indices
        for i in range(1, batch_size):
            variable[i, 0] = self.src_vocabulary.get_eos()
        variable = variable.transpose(0, 1)
        variable = variable.cuda() if use_cuda else variable
        lengths = [len(indices)]
        lengths += [1 for _ in range(self.batch_size-1)]
        return variable, lengths
        
    def prepare_noisy_input(self, batch: Batch):
        new_src_variable, new_src_lengths = self.prepare_noisy_variable(batch.src_variable)
        new_tgt_variable, new_tgt_lengths = self.prepare_noisy_variable(batch.tgt_variable)
        return Batch(new_src_variable, new_tgt_variable, new_src_lengths, new_tgt_lengths)
    
    def prepare_translated_noisy_input(self, batch: Batch):
        new_src_variable, _ = self.prepare_translated_variable(batch.src_variable, lang="src")
        new_src_variable, new_src_lengths = self.prepare_noisy_variable(new_src_variable)
        
        new_tgt_variable, _ = self.prepare_translated_variable(batch.tgt_variable, lang="tgt")
        new_tgt_variable, new_tgt_lengths = self.prepare_noisy_variable(new_tgt_variable)
        return Batch(new_src_variable, new_tgt_variable, new_src_lengths, new_tgt_lengths)
            
    def prepare_noisy_variable(self, variable):
        assert variable.size(1) == self.batch_size
        max_length = variable.size(0)
        variable = variable.transpose(0, 1)
        new_lengths = []
        new_varibale = Variable(torch.zeros(self.batch_size, max_length)).type(torch.LongTensor)
        for b in range(self.batch_size):
            indices = [elem for elem in variable[b].data if elem != 0][:-1]
            noisy = add_noise(indices) + [2, ]
            new_lengths.append(len(noisy))
            noisy = noisy + [0 for _ in range(max_length - len(noisy))]
            new_varibale[b] = torch.LongTensor(noisy)
        return new_varibale.transpose(0, 1), new_lengths
    
    def prepare_translated_variable(self, variable, lang="src"):
        variable = variable.transpose(0, 1)
        new_sentences = []
        for b in range(self.batch_size):
            if lang == "src":
                translated = self.current_model.translate_src2tgt(list(variable[b].data))
            else:
                translated = self.current_model.translate_tgt2src(list(variable[b].data))
            new_sentences.append(translated)
        lengths = [len(sentence) for sentence in new_sentences]
        max_length = max(lengths)
        new_variable = Variable(torch.zeros(self.batch_size, max_length)).type(torch.LongTensor)
        for b in range(self.batch_size):
            current_sentence = new_sentences[b] 
            current_sentence = current_sentence + [0 for _ in range(max_length-len(current_sentence))]
            new_variable[b] = torch.LongTensor(current_sentence)
        return new_variable.transpose(0, 1), lengths
            
    @staticmethod
    def add_noise(sequence, drop_probability=0.1, shuffle_max_distance=3):
        new_sequence = [elem for elem in sequence if np.random.random_sample() > drop_probability]
        new_sequence = [x for i, x in sorted(enumerate(new_sequence), key = lambda x: x[0] + (shuffle_max_distance+1)*np.random.random())]
        return new_sequence

In [130]:
VAL_INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.en"
VAL_OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.de"
val_batch_generator = BatchGenerator([(VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME), ], 64, 50,
                                         M0.src_vocabulary, M0.tgt_vocabulary, use_cuda)
val_batches = []
for batch in val_batch_generator:
    val_batches.append(batch)
val_perm = np.random.permutation(len(val_batches))[:1000]
val_batches = [batch for i, batch in enumerate(val_batches) if i in val_perm]   
batch = val_batches[900]


val-sorted.en:   0%|          | 0.00/23.9M [00:00<?, ?B/s]
val-sorted.en:   4%|▍         | 1.05M/23.9M [00:00<00:20, 1.09MB/s]
val-sorted.en:   9%|▉         | 2.10M/23.9M [00:01<00:17, 1.23MB/s]
val-sorted.en:  13%|█▎        | 3.15M/23.9M [00:02<00:14, 1.39MB/s]
val-sorted.en:  18%|█▊        | 4.19M/23.9M [00:02<00:12, 1.57MB/s]
val-sorted.en:  22%|██▏       | 5.24M/23.9M [00:02<00:10, 1.75MB/s]
val-sorted.en:  26%|██▋       | 6.29M/23.9M [00:03<00:09, 1.92MB/s]
val-sorted.en:  31%|███       | 7.34M/23.9M [00:03<00:08, 2.05MB/s]
val-sorted.en:  35%|███▌      | 8.39M/23.9M [00:04<00:06, 2.22MB/s]
val-sorted.en:  39%|███▉      | 9.44M/23.9M [00:04<00:06, 2.41MB/s]
val-sorted.en:  44%|████▍     | 10.5M/23.9M [00:04<00:05, 2.57MB/s]
val-sorted.en:  48%|████▊     | 11.5M/23.9M [00:05<00:04, 2.77MB/s]
val-sorted.en:  53%|█████▎    | 12.6M/23.9M [00:05<00:04, 2.80MB/s]
val-sorted.en:  57%|█████▋    | 13.6M/23.9M [00:05<00:03, 3.01MB/s]
val-sorted.en:  61%|██████▏   | 14.7M/23.9M [00:06<00:02

In [131]:
print(batch)

Batch: Variable containing:
     3      3      3  ...       3      3    810
     3      3    816  ...       3      3   3036
     3      3      3  ...    1691  42427      7
        ...            ⋱           ...         
  2227    475  13011  ...       0      0      0
     3      3      3  ...       0      0      0
     2      2      2  ...       0      0      0
[torch.LongTensor of size 28x64]
, Variable containing:
     3      3      3  ...       3      3   2405
     3     75   2421  ...       3      3      3
   660      3      4  ...       3  47922     13
        ...            ⋱           ...         
     0      0      0  ...       0      0      0
     0      0      0  ...       0      0      0
     0      0      0  ...       0      0      0
[torch.LongTensor of size 36x64]
, [28, 28, 28, 27, 27, 27, 27, 27, 27, 26, 26, 26, 26, 26, 26, 26, 25, 25, 25, 25, 25, 25, 25, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 22, 22, 22, 22, 22, 21, 21, 21, 

In [132]:
state = GlobalState()
print(state.prepare_translated_noisy_input(batch))

Batch: Variable containing:
     3      3      3  ...       3  47922     13
     3      3      3  ...       3      3   2406
     3      3     90  ...       3      3   3896
        ...            ⋱           ...         
   130      2      0  ...       0      0      0
     2      0      0  ...       0      0      0
     0      0      0  ...       0      0      0
[torch.LongTensor of size 28x64]
, Variable containing:
     3     27   8604  ...       3      3    810
     3      3      3  ...       3      3      3
 15052      3      4  ...      10  42427      3
        ...            ⋱           ...         
     0      0      0  ...       0      0      0
     0      0      0  ...       0      0      0
     0      0      0  ...       0      0      0
[torch.LongTensor of size 36x64]
, [27, 26, 24, 25, 26, 27, 23, 23, 23, 25, 25, 20, 22, 23, 23, 24, 25, 23, 22, 24, 23, 24, 22, 22, 23, 19, 20, 22, 22, 21, 21, 23, 18, 21, 22, 21, 20, 20, 21, 22, 19, 21, 18, 23, 20, 21, 21, 18, 21, 18, 19, 18, 